# 01 - Document Preprocessing

## Project Overview

This notebook is the first stage of the **Multi-Course Question Answering System**,
a Retrieval-Augmented Generation (RAG) pipeline that answers student questions
strictly from university course materials.

**Goal of this notebook:** turn the raw files sitting under `data/<course>/`
(PDF, DOCX, TXT, CSV) into clean, appropriately sized text chunks with rich
metadata, and persist them to disk so that Notebook 2 can embed them without
having to re-parse the original files.

Pipeline position:

```
Course Materials -> [THIS NOTEBOOK] -> Processed Chunks -> Embeddings -> ChromaDB -> Retriever -> LLM
```


## Imports

In [11]:
import sys
from pathlib import Path

# Make the project's shared `rag_core` package importable when the notebook
# is launched from the `notebooks/` folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from rag_core import AppConfig, DocumentProcessor
from rag_core.document_processor import RawDocument, ProcessedChunk

print(f"Project root: {PROJECT_ROOT}")


Project root: f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant


## Configuration

All tunable parameters (chunk size, chunk overlap, supported extensions,
course names, storage paths, ...) live in a single `AppConfig` object so
nothing is hard-coded inside this notebook.

In [12]:
config = AppConfig()
config.project_root = PROJECT_ROOT
config.__post_init__()   # recompute derived paths against the resolved project root
config.ensure_directories()

print("Data directory      :", config.data_dir)
print("Persist directory    :", config.persist_directory)
print("Supported extensions :", config.supported_extensions)
print("Course names          :", config.course_names)
print("Chunk size / overlap  :", config.chunk_size, "/", config.chunk_overlap)


Data directory      : f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\data
Persist directory    : f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db
Supported extensions : ('.pdf', '.docx', '.txt', '.csv')
Course names          : ('Artificial Intelligence', 'Machine Learning', 'Deep Learning')
Chunk size / overlap  : 1000 / 150


## Load Documents

`DocumentProcessor.discover_files()` walks every `data/<course>/` folder and
returns the files whose extension is supported. Anything else is skipped
with a warning log rather than raising an error, so a stray `.jpg` or
`.pptx` in a course folder will not break the pipeline.

In [13]:
processor = DocumentProcessor(config)

discovered_files = processor.discover_files()
print(f"\nTotal supported files discovered: {len(discovered_files)}")
for path in discovered_files:
    print(" -", path.relative_to(config.data_dir))


2026-08-02 01:02:24 | INFO     | DocumentProcessor | Discovered 13 supported file(s).

Total supported files discovered: 13
 - Artificial Intelligence\expert_systems.docx
 - Artificial Intelligence\glossary.txt
 - Artificial Intelligence\grades.csv
 - Artificial Intelligence\intro_to_ai.pdf
 - Artificial Intelligence\search_algorithms.pdf
 - Deep Learning\cnn_architectures.docx
 - Deep Learning\glossary.txt
 - Deep Learning\grades.csv
 - Deep Learning\neural_networks_basics.pdf
 - Machine Learning\glossary.txt
 - Machine Learning\grades.csv
 - Machine Learning\supervised_learning.docx
 - Machine Learning\unsupervised_learning.pdf


## Document Validation

Before chunking, we load every discovered file through the format-specific
loader (`_load_pdf`, `_load_docx`, `_load_txt`, `_load_csv`). Corrupted or
empty files are logged and skipped rather than crashing the run — this is
handled inside `DocumentProcessor.load_file`.

In [14]:
raw_documents: list[RawDocument] = []
failed_files = []

for path in discovered_files:
    docs = processor.load_file(path)
    if docs:
        raw_documents.extend(docs)
    else:
        failed_files.append(path)

print(f"Successfully loaded sections/pages: {len(raw_documents)}")
print(f"Files that produced no content    : {len(failed_files)}")
for f in failed_files:
    print(" -", f)


Successfully loaded sections/pages: 13
Files that produced no content    : 0


## Text Cleaning

`DocumentProcessor.clean_text` performs:

- Unicode normalisation (NFKC)
- Whitespace normalisation (tabs/multiple spaces collapsed)
- Removal of duplicate blank lines
- Paragraph preservation (single newlines kept as paragraph breaks)

Let's inspect the effect of cleaning on a sample document.

In [15]:
sample_raw = raw_documents[0]
sample_cleaned = processor.clean_text(sample_raw.text)

print("--- BEFORE CLEANING (first 300 chars) ---")
print(repr(sample_raw.text[:300]))
print("\n--- AFTER CLEANING (first 300 chars) ---")
print(repr(sample_cleaned[:300]))


--- BEFORE CLEANING (first 300 chars) ---
'Expert Systems\nExpert systems are AI programs that emulate the decision-making ability of a human expert by relying on a knowledge base of facts and an inference engine that applies logical rules to that knowledge base.\nComponents\nThe two core components of an expert system are the knowledge base, w'

--- AFTER CLEANING (first 300 chars) ---
'Expert Systems\nExpert systems are AI programs that emulate the decision-making ability of a human expert by relying on a knowledge base of facts and an inference engine that applies logical rules to that knowledge base.\nComponents\nThe two core components of an expert system are the knowledge base, w'


## Chunking

Documents are split with LangChain's `RecursiveCharacterTextSplitter`, using
a separator hierarchy (`\n\n`, `\n`, `. `, ` `, `""`) so the splitter prefers
breaking on paragraph and sentence boundaries before falling back to hard
character cuts. Chunk size and overlap come from `AppConfig`.

In [16]:
processed_chunks: list[ProcessedChunk] = processor.chunk_documents(raw_documents)

print(f"Total chunks created: {len(processed_chunks)}")


2026-08-02 01:02:25 | INFO     | DocumentProcessor | Created 13 chunk(s) from 13 document page(s)/section(s).
Total chunks created: 13


## Metadata Creation

Every chunk carries the metadata required for source attribution later in
the pipeline: **Course Name, File Name, Page Number, Chunk ID, Document
Type**. This happens automatically inside `chunk_documents`, driven by the
`RawDocument` each piece was split from.

In [17]:
sample_chunk = processed_chunks[0]
print("chunk_id     :", sample_chunk.chunk_id)
print("course       :", sample_chunk.course)
print("file_name    :", sample_chunk.file_name)
print("doc_type     :", sample_chunk.doc_type)
print("page_number  :", sample_chunk.page_number)
print("char_count   :", sample_chunk.char_count)
print("\nmetadata dict:", sample_chunk.to_metadata())


chunk_id     : Artificial_Intelligence__expert_systems__p0__c00001
course       : Artificial Intelligence
file_name    : expert_systems.docx
doc_type     : docx
page_number  : None
char_count   : 416

metadata dict: {'chunk_id': 'Artificial_Intelligence__expert_systems__p0__c00001', 'course': 'Artificial Intelligence', 'file_name': 'expert_systems.docx', 'doc_type': 'docx', 'page_number': -1, 'char_count': 416}


## Chunk Preview

A quick look at a handful of chunks across different courses and file types.

In [18]:
import random

random.seed(42)
preview_chunks = random.sample(processed_chunks, k=min(5, len(processed_chunks)))

for i, chunk in enumerate(preview_chunks, start=1):
    print(f"[{i}] {chunk.course} | {chunk.file_name} | page={chunk.page_number} | {chunk.char_count} chars")
    print(chunk.text[:220].replace("\n", " "))
    print("-" * 100)


[1] Machine Learning | grades.csv | page=None | 476 chars
StudentID, Assignment, Score, MaxScore StudentID: 201; Assignment: Regression Lab; Score: 85; MaxScore: 100 StudentID: 202; Assignment: Regression Lab; Score: 79; MaxScore: 100 StudentID: 203; Assignment: Regression Lab;
----------------------------------------------------------------------------------------------------
[2] Artificial Intelligence | glossary.txt | page=None | 596 chars
AI Glossary  Agent: An entity that perceives its environment through sensors and acts upon it through actuators.  Heuristic: A technique that helps an algorithm find a good-enough solution faster, without guaranteeing op
----------------------------------------------------------------------------------------------------
[3] Artificial Intelligence | expert_systems.docx | page=None | 416 chars
Expert Systems Expert systems are AI programs that emulate the decision-making ability of a human expert by relying on a knowledge base of facts and an infer

## Statistics

Basic sanity statistics over the processed corpus, broken down by course and document type.

In [19]:
from collections import Counter

by_course = Counter(c.course for c in processed_chunks)
by_type = Counter(c.doc_type for c in processed_chunks)
lengths = [c.char_count for c in processed_chunks]

print("Chunks per course:")
for course, count in by_course.items():
    print(f"  {course:<25} {count}")

print("\nChunks per document type:")
for doc_type, count in by_type.items():
    print(f"  {doc_type:<10} {count}")

if lengths:
    print(f"\nChunk length (chars) - min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)/len(lengths):.1f}")


Chunks per course:
  Artificial Intelligence   5
  Deep Learning             4
  Machine Learning          4

Chunks per document type:
  docx       3
  txt        3
  csv        3
  pdf        4

Chunk length (chars) - min: 335, max: 841, avg: 542.8


## Save Processed Documents

Chunks are persisted as JSON Lines under `chroma_db/processed_chunks.jsonl`
so Notebook 2 can load them directly without re-parsing PDFs/DOCX/TXT/CSV
files.

In [20]:
output_path = processor.save_chunks(processed_chunks)
print("Processed chunks saved to:", output_path)


2026-08-02 01:02:25 | INFO     | DocumentProcessor | Saved 13 chunk(s) to f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db\processed_chunks.jsonl
Processed chunks saved to: f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db\processed_chunks.jsonl


## Next Step

Continue to **`02_Vector_Database.ipynb`** to embed these chunks and build
the persisted ChromaDB collection.